# YOLO 2회차 실습 A — 컨베이어 상자 '크기별' 자동 집계

택배 분류장 컨베이어 CCTV 영상에서 **지나간 상자를 소형·중형·대형(S/M/L)으로 나눠 세는 기능**을 처음부터 만들어 봅니다.
단순히 개수만 세는 것이 아니라, 세그멘테이션(segmentation, 물체를 픽셀 단위 모양 그대로 찾아내는 방식) 마스크(mask, 물체가 차지한 픽셀 영역)로 상자의 실제 크기를 재고 크기 기준을 데이터로 정하는 데까지 갑니다.
각 단계는 앞 단계의 결과물을 그대로 이어 받아 기능을 하나씩 더합니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 사전학습(pretrained, 공개 대규모 데이터로 미리 학습된) 모델 그대로 써 보기 — 무엇을 못 하는지 확인 | 탐지 결과 비교 |
| Step 2 | 세그멘테이션 모델 파인튜닝(fine-tuning, 사전학습 모델을 우리 데이터로 추가 학습) + 성능 평가 (박스·마스크 mAP(평균 정밀도 — 탐지 성능 종합 점수), 임계값(threshold, 이 값 미만 예측은 버리는 기준), 오류 분석) | `box_seg_best.pt`, 평가 지표 표 |
| Step 3 | 트래킹(tracking, 추적) — 프레임(frame, 영상을 이루는 낱장 이미지)이 바뀌어도 같은 상자에 같은 ID(물체마다 붙이는 고유 번호) 붙이기 | `tracked.mp4` |
| Step 4 | 라인 카운팅 — 기준선을 넘은 상자 수 세기 | `counting.mp4`, `box_count_log.csv` |
| Step 5 | 크기 분류 — 마스크로 크기를 재고, S/M/L 기준을 데이터로 정해 크기별로 세기 | `size_counting.mp4`, `box_size_log.csv`, `size_rules.json` |

**데이터셋**: Ultralytics(YOLO를 만든 회사·라이브러리)에서 공개한 *Package Segmentation* 데이터셋 (택배 분류장 CCTV 이미지, 클래스(class, 모델이 구분하는 물체 종류) 1개 `package`, 학습 1,920장 · 검증 188장 · 테스트 89장, 약 100MB)

**실행 환경**: Google Colab, 런타임(Colab이 빌려주는 실행 환경) 유형 **T4 GPU**(딥러닝 계산을 빠르게 하는 그래픽 처리 장치) (메뉴 → 런타임 → 런타임 유형 변경)

> 결과 파일은 Google Drive의 `MyDrive/yolo_day2` 폴더에 저장됩니다. Colab 런타임이 끊겨도 학습한 모델이 남고, 마지막 통합 실습(`04_yolo_day2_monitor_lab_v1.1.ipynb`)에서 이 파일들을 다시 사용합니다.

## 0. 준비

In [ ]:
# ultralytics: YOLO 라이브러리 / lap: 트래커 짝짓기 계산 / shapely: 다각형 계산
!pip -q install ultralytics lap shapely

In [ ]:
# 파이썬 기본 도구 — 파일·경로·시간·정규식 등
import os, json, shutil, base64, subprocess, re, time, glob
from pathlib import Path
from collections import Counter, defaultdict  # Counter: 개수 세기 / defaultdict: 기본값 딕셔너리

# 이미지·영상 처리, 배열 계산, 표, 그래프, 딥러닝 프레임워크
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Image, display  # 노트북 안에 영상·그림 표시
from ultralytics import YOLO  # 모델 불러오기·학습·추론·트래킹 핵심 클래스

# GPU 확인 — False면 런타임 유형을 T4 GPU로 바꿀 것
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 결과 저장 위치 — Google Drive를 연결하면 런타임이 끊겨도 결과가 남습니다
USE_DRIVE = True  # False로 바꾸면 런타임 안 outputs 폴더에 저장 (끊기면 사라짐)

# google.colab 모듈을 불러올 수 있으면 Colab 환경으로 판단
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')            # 처음 한 번 Google 계정 접근 허용 창이 뜹니다
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day2')  # 통합 실습도 이 폴더를 씀
else:
    SAVE_DIR = Path('outputs')
SAVE_DIR.mkdir(parents=True, exist_ok=True)  # 상위 폴더까지 생성, 이미 있으면 그냥 통과
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# 학습 설정 — 시간이 부족하면 QUICK_RUN = True (학습을 짧게 줄입니다)
QUICK_RUN = False  # True: 3 epoch·416px로 빨리 끝나지만 정확도는 낮아짐

# epochs: 전체 데이터 반복 횟수 / imgsz: 입력 이미지 크기(px) / batch: 한 번에 넣는 장수
CFG = dict(epochs=20, imgsz=640, batch=16) if not QUICK_RUN else dict(epochs=3, imgsz=416, batch=16)
# 이후 학습·평가·추론·트래킹 모두 CFG 값을 같이 씀
print(CFG)

In [ ]:
# 화면 표시용 도우미 함수
# show_bgr: OpenCV 이미지(BGR) 한 장을 화면에 표시 (반환값 없음)
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')  # [..., ::-1]: 색 순서 BGR→RGB로 뒤집기
    if title:
        plt.title(title)
    plt.show()

# show_grid: 여러 이미지를 cols열 격자로 한 번에 표시
def show_grid(imgs, titles=None, cols=2, size=4.5):
    rows = (len(imgs) + cols - 1) // cols  # 필요한 행 수 (올림 나눗셈)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):  # 격자 칸을 1줄로 펴서 차례로 채움
        ax.axis('off')
        if i < len(imgs):  # 이미지가 없는 남는 칸은 빈칸으로
            ax.imshow(imgs[i][..., ::-1])
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

# show_video: mp4 경로 → 노트북 안에서 재생 (반환값 없음)
def show_video(path, width=360):
    """mp4 영상을 브라우저에서 재생 가능한 H.264로 바꿔 노트북에 표시"""
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))  # 변환본 파일 이름: 원본이름_h264.mp4
    # OpenCV가 만든 mp4v 영상은 브라우저 재생 불가 → ffmpeg로 H.264 재인코딩
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()  # 영상 파일을 문자열로 바꿔 HTML에 직접 넣음
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

# save_out: 파일 경로 → SAVE_DIR에 복사한 경로
def save_out(path):
    """결과 파일을 저장 폴더(SAVE_DIR)에 복사"""
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():  # 원본과 대상이 같은 파일이면 복사 생략
        shutil.copy(path, dst)
    return dst

---
## Step 1. 사전학습 모델 그대로 써 보기

### 1-1. 데이터셋 내려받기와 라벨 구조 확인

`check_det_dataset()`은 데이터셋 설정 파일(`package-seg.yaml` — 데이터 위치와 클래스 이름을 적은 YAML 형식 파일)을 읽고, 데이터가 없으면 자동으로 내려받습니다.

In [ ]:
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset('package-seg.yaml')  # 설정 읽기 + 데이터 없으면 자동 다운로드 → 딕셔너리
DATA_ROOT = Path(data['path'])  # 데이터셋 최상위 폴더 (images/, labels/ 아래 분할별 폴더)
print('데이터 위치:', DATA_ROOT)
print('클래스:', data['names'])  # 클래스 번호 → 이름 대응표
# 분할마다 이미지 수와 라벨 수가 맞는지 확인
for split in ['train', 'val', 'test']:
    n_img = len(list((DATA_ROOT / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split:5s}: 이미지 {n_img:5d}장 / 라벨 {n_lbl:5d}개')  # :5s, :5d → 5칸 폭으로 맞춰 출력

라벨(label, 사람이 표시해 둔 정답) 파일 한 줄은 상자 하나입니다. 이 데이터셋은 상자 테두리를 **다각형 꼭짓점 좌표**로 적어 두었습니다. 사각형 박스보다 정보가 많아서, 뒤에서 상자의 실제 크기를 잴 때 이 정보를 씁니다.

```
클래스번호  x1 y1  x2 y2  x3 y3 ...   ← 모두 0~1로 정규화된 좌표
```

In [ ]:
# read_gt: 이미지 경로 → 라벨 파일을 읽어 정답 박스·다각형(픽셀 좌표) 반환
def read_gt(img_path):
    """이미지 경로 → (정답 박스 (N,4) [x1,y1,x2,y2], 정답 다각형 목록) 픽셀 좌표"""
    img_path = Path(img_path)
    lbl = img_path.parent.parent.parent / 'labels' / img_path.parent.name / (img_path.stem + '.txt')  # images/val/a.jpg → labels/val/a.txt
    h, w = cv2.imread(str(img_path)).shape[:2]  # 이미지 높이·폭 (픽셀)
    boxes, polys = [], []
    if lbl.exists():
        for line in lbl.read_text().strip().splitlines():
            v = np.array(line.split()[1:], float)  # 맨 앞 클래스 번호를 빼고 좌표만
            xs, ys = v[0::2] * w, v[1::2] * h  # 짝수 번째=x, 홀수 번째=y, 0~1 → 픽셀
            polys.append(np.stack([xs, ys], 1))  # (꼭짓점 수, 2) 모양 다각형
            boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])  # 다각형을 감싸는 사각형 [x1,y1,x2,y2]
    return np.array(boxes).reshape(-1, 4), polys  # 상자가 0개여도 (0,4) 모양 유지

# draw_boxes: 박스마다 사각형(+글자)을 그려 같은 이미지 반환
def draw_boxes(img, boxes, color, label=None, thick=2):
    for b in boxes:
        x1, y1, x2, y2 = map(int, b[:4])  # 앞 4개 값(좌상단·우하단)을 정수 픽셀로
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)
        if label:
            cv2.putText(img, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)  # 박스 위에 글자, 화면 밖으로 안 나가게
    return img

# draw_polys: 다각형을 반투명하게 칠하고 테두리를 그린 이미지 반환
def draw_polys(img, polys, color, alpha=0.35):
    overlay = img.copy()  # 칠할 사본 — 나중에 원본과 섞음
    for p in polys:
        cv2.fillPoly(overlay, [p.astype(np.int32)], color)
        cv2.polylines(img, [p.astype(np.int32)], True, color, 2)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)  # 사본 35% + 원본 65% 섞기 → 반투명

# 검증 이미지 목록과 고정 샘플 4장 (매번 같은 이미지)
VAL_IMAGES = sorted((DATA_ROOT / 'images' / 'val').glob('*.jpg'))
SAMPLES = [VAL_IMAGES[i] for i in (3, 40, 95, 150)]

# 샘플마다 정답 다각형(초록)과 감싸는 박스(주황)를 그려 비교
gt_imgs = []
for p in SAMPLES:
    b, poly = read_gt(p)
    gt_imgs.append(draw_boxes(draw_polys(cv2.imread(str(p)), poly, (0, 255, 0)), b, (0, 200, 255), thick=1))
show_grid(gt_imgs, [f'GT: {len(read_gt(p)[1])} packages (polygon + box)' for p in SAMPLES])

초록색이 다각형 정답, 주황색 얇은 선이 다각형을 감싸는 사각형입니다. **비스듬히 놓인 상자는 사각형이 실제 상자보다 훨씬 크게** 잡히는 것을 확인하세요.

### 1-2. COCO 사전학습 모델로 탐지해 보기

`yolo26n.pt`(`.pt` = 학습된 가중치 파일, `n` = 가장 작은 크기)는 COCO 데이터셋(사람·자동차·가방 등 80개 클래스)으로 학습된 모델입니다. 택배 상자도 찾을 수 있을까요?

In [ ]:
coco_model = YOLO('yolo26n.pt')  # COCO 80클래스 사전학습 탐지 모델 (n = 가장 작은 크기)

# [0]: 이미지 1장의 결과 / conf=0.25: 신뢰도 0.25 미만 버림 / .plot(): 결과 그림
pred_imgs = [coco_model(str(p), verbose=False, conf=0.25)[0].plot() for p in SAMPLES]
show_grid(pred_imgs, ['COCO pretrained'] * len(SAMPLES))

In [ ]:
# 검증 이미지 전체에서 어떤 클래스로 탐지하는지 세어 봅니다
cls_counter, no_det, total_gt = Counter(), 0, 0  # 클래스별 탐지 수, 빈 이미지 수, 정답 상자 수
for p in VAL_IMAGES:
    r = coco_model(str(p), verbose=False, conf=0.25)[0]
    names = [r.names[int(c)] for c in r.boxes.cls]  # r.boxes.cls: 클래스 번호 → 이름으로 변환
    cls_counter.update(names)
    no_det += (len(names) == 0)  # 탐지 0개면 True(=1)가 더해짐
    total_gt += len(read_gt(p)[0])  # 이 이미지의 정답 박스 수

print(f'검증 이미지 {len(VAL_IMAGES)}장, 정답 상자 {total_gt}개')
print(f'아무것도 찾지 못한 이미지: {no_det}장')
pd.DataFrame(cls_counter.most_common(10), columns=['coco_class', 'count'])  # 많이 나온 클래스 10개 표 (마지막 줄이라 화면 출력)

COCO에는 '택배 상자' 클래스가 없습니다. 그래서 상자는 거의 찾지 못하고, 긴 컨베이어 벨트를 **`train`(기차)** 으로 착각하는 일이 많습니다.
'물체처럼 보이는 것'을 찾는 능력은 이미 갖고 있으니, **우리 대상의 이름과 모양**만 우리 데이터로 가르치면 됩니다. 이것이 Step 2의 파인튜닝입니다.

---
## Step 2. 세그멘테이션 모델 파인튜닝과 성능 평가

### 2-1. 왜 세그멘테이션 모델인가

탐지(detect, 물체 위치를 사각형 박스로 찾는 방식) 모델은 사각형 박스만 돌려줍니다. 세그멘테이션(segment) 모델은 박스와 함께 **물체 모양 그대로의 마스크**를 돌려줍니다.
상자 크기를 재려면 마스크가 필요합니다. 비스듬한 상자의 박스 면적은 실제보다 크게 나오기 때문입니다.

| 모델 | 가중치(weights, 학습된 모델 값이 담긴 파일) | 출력 | 이번 실습에서 쓰는 곳 |
|---|---|---|---|
| 탐지 | `yolo26n.pt` | 박스 | Step 1 비교용 |
| 세그멘테이션 | `yolo26n-seg.pt` | 박스 + 마스크 | Step 2~5 전체 |

### 2-2. 학습

사전학습 가중치(`yolo26n-seg.pt`)에서 출발해 상자 데이터로 추가 학습합니다. T4 GPU 기준 20 epoch(에포크 — 학습 데이터 전체를 한 번 다 학습하는 단위)에 15~20분 정도 걸립니다.

In [ ]:
model = YOLO('yolo26n-seg.pt')  # COCO 사전학습 세그멘테이션 모델 — 학습 출발점
t0 = time.time()  # 학습 시간 측정 시작
# 파인튜닝: 결과는 runs_box/seg_baseline, 같은 폴더 있으면 덮어씀(exist_ok), seed=0 고정
model.train(data='package-seg.yaml', epochs=CFG['epochs'], imgsz=CFG['imgsz'], batch=CFG['batch'],
            project='runs_box', name='seg_baseline', exist_ok=True, seed=0, verbose=False)
TRAIN_MIN = (time.time() - t0) / 60  # 학습 시간(분) — 2-7 실험 표에 기록
RUN_DIR = Path(model.trainer.save_dir)  # 실제로 저장된 학습 결과 폴더
BEST = RUN_DIR / 'weights' / 'best.pt'  # 검증 점수가 가장 좋았던 epoch의 가중치
print(f'학습 시간: {TRAIN_MIN:.1f}분')
print('최고 성능 가중치:', BEST)
shutil.copy(BEST, SAVE_DIR / 'box_seg_best.pt')  # Drive 폴더에 복사 (통합 실습에서 사용)
print('저장 폴더에 복사:', SAVE_DIR / 'box_seg_best.pt')

In [ ]:
# epoch별 손실과 지표 변화
display(Image(filename=str(RUN_DIR / 'results.png'), width=900))  # Ultralytics가 학습 중 자동 저장한 그래프

그래프 읽는 법
- **train/val loss**(loss = 손실, 예측이 정답과 다른 정도 — 작을수록 좋음 / train = 학습 세트, val = 검증 세트)가 함께 내려가면 정상적으로 배우는 중입니다. train만 내려가고 val이 올라가면 과적합(overfitting, 학습 데이터만 외워 새 데이터에서 성능이 떨어지는 상태)입니다.
- 세그멘테이션 모델은 손실이 하나 더 있습니다. **seg_loss**는 마스크 모양이 얼마나 정답과 다른지를 나타냅니다.
- **metrics/mAP50(B)** 는 박스, **metrics/mAP50(M)** 은 마스크 기준 점수입니다.

### 2-3. 성능 지표 확인

| 지표 | 질문 | 계산 |
|---|---|---|
| Precision (정밀도) | 상자라고 한 것 중 진짜 상자는 몇 %? | TP / (TP + FP) |
| Recall (재현율) | 진짜 상자 중 찾아낸 것은 몇 %? | TP / (TP + FN) |
| mAP50 | IoU(겹침 비율) 0.5 기준으로 본 종합 점수 | 정밀도-재현율 곡선 아래 면적 |
| mAP50-95 | 위치·모양까지 얼마나 정확한가 | IoU 0.5~0.95 구간 평균 |

TP(True Positive, 맞게 찾은 상자) · FP(False Positive, 없는 상자를 찾은 헛탐지) · FN(False Negative, 있는데 놓친 상자)

IoU(Intersection over Union)는 예측과 정답이 겹치는 비율입니다. 박스 지표는 사각형끼리, **마스크 지표는 픽셀 단위 모양끼리** 겹침을 계산하므로 마스크 점수가 더 엄격합니다.

In [ ]:
best_model = YOLO(BEST)  # 학습한 최고 가중치 — Step 5까지 이 모델 사용

# evaluate: 모델, 분할 이름('val'/'test') → (지표 객체, 요약 딕셔너리)
def evaluate(weights_model, split):
    m = weights_model.val(data='package-seg.yaml', split=split, imgsz=CFG['imgsz'], batch=CFG['batch'],
                          project='runs_box', name=f'val_{split}', exist_ok=True, verbose=False)
    # box.mp/mr: 박스 Precision/Recall, map50·map: mAP50·mAP50-95, seg: 마스크 기준
    return m, {'split': split,
               'box_precision': round(m.box.mp, 3), 'box_recall': round(m.box.mr, 3),
               'box_map50': round(m.box.map50, 3), 'box_map50_95': round(m.box.map, 3),
               'mask_map50': round(m.seg.map50, 3), 'mask_map50_95': round(m.seg.map, 3)}

# 검증(val)과 테스트(test) 세트 각각 평가 후 한 표로
val_metrics, val_row = evaluate(best_model, 'val')
test_metrics, test_row = evaluate(best_model, 'test')
metrics_table = pd.DataFrame([val_row, test_row])
metrics_table

학습에 쓰지 않은 **test** 세트 점수가 val과 비슷하면, 모델이 특정 이미지만 외운 것이 아니라 일반화(처음 보는 데이터에서도 비슷한 성능을 내는 것)되었다는 뜻입니다.

In [ ]:
VAL_DIR = Path(val_metrics.save_dir)  # val 평가 결과가 저장된 폴더
for name in ['BoxPR_curve.png', 'MaskPR_curve.png']:  # 박스·마스크 기준 정밀도-재현율(PR) 곡선
    if (VAL_DIR / name).exists():  # 파일이 있을 때만 표시
        display(Image(filename=str(VAL_DIR / name), width=480))

### 2-4. 박스와 마스크, 면적이 얼마나 다른가

검증 이미지의 모든 예측에 대해 **박스 면적 ÷ 마스크 면적** 을 계산합니다. 1에 가까우면 박스가 상자를 꼭 맞게 감싼 것이고, 클수록 상자가 비스듬해 박스에 빈 공간이 많은 것입니다.

In [ ]:
ratios, worst = [], None  # 면적비 목록, 차이가 가장 큰 예측
for p in VAL_IMAGES:
    r = best_model(str(p), conf=0.4, imgsz=CFG['imgsz'], verbose=False)[0]  # 신뢰도 0.4 이상 예측만
    if r.masks is None:  # 탐지가 하나도 없으면 masks가 None
        continue
    for b, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy):  # xyxy: 박스 [x1,y1,x2,y2] / masks.xy: 마스크 윤곽선 좌표
        if len(poly) < 3:  # 꼭짓점 3개 미만이면 면적 계산 불가 → 건너뜀
            continue
        mask_area = cv2.contourArea(poly.astype(np.float32))  # 윤곽선 내부 면적 = 마스크 면적 (px²)
        box_area = (b[2] - b[0]) * (b[3] - b[1])  # 박스 면적 = 폭 × 높이
        if mask_area > 500:  # 너무 작은 마스크는 비율이 튀므로 제외
            ratio = box_area / mask_area
            ratios.append(ratio)
            if worst is None or ratio > worst[0]:  # 가장 큰 비율의 예측을 기억
                worst = (ratio, p, b, poly)

ratios = np.array(ratios)
print(f'예측 {len(ratios)}개 | 박스/마스크 면적비 중앙값 {np.median(ratios):.2f} | '
      f'1.5배 이상인 예측 {np.mean(ratios >= 1.5) * 100:.0f}%')

# 차이가 가장 큰 예측: 빨강 = 박스, 초록 = 마스크
ratio, p, b, poly = worst
im = draw_polys(cv2.imread(str(p)), [poly], (0, 255, 0))
draw_boxes(im, [b], (0, 0, 255), thick=2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ratios, bins=30, color='tab:blue'); axes[0].axvline(1.5, ls='--', c='gray')  # 비율 분포 + 1.5배 기준선
axes[0].set_xlabel('box area / mask area'); axes[0].set_ylabel('count')
axes[1].imshow(im[..., ::-1]); axes[1].axis('off')  # BGR→RGB로 뒤집어 표시
axes[1].set_title(f'largest gap: box is {ratio:.1f}x the mask (red = box, green = mask)', fontsize=9)
plt.tight_layout(); plt.show()

박스 면적으로 크기를 나누면, 같은 상자라도 **놓인 각도에 따라 S가 되었다가 L이 되었다가** 합니다. 그래서 Step 5에서는 마스크 면적을 씁니다.

### 2-5. 신뢰도 임계값 정하기 — 직접 TP/FP/FN 세어 보기

mAP는 모든 임계값을 종합한 점수입니다. 하지만 실제 서비스에서는 **임계값(conf) 하나를 골라야**(conf = confidence, 신뢰도 — 모델이 예측마다 매긴 0~1 확신 점수) 합니다.
검증 이미지마다 예측을 한 번만 해 두고, 임계값을 바꿔 가며 TP·FP·FN을 직접 세어 봅니다.

In [ ]:
# box_iou: 박스 a (N,4), b (M,4) → 모든 짝의 IoU (N,M) 행렬
def box_iou(a, b):
    """a: (N,4), b: (M,4) → (N,M) IoU 행렬"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])  # 겹친 영역 좌상단
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])  # 겹친 영역 우하단
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)  # np.clip: 음수(안 겹침)는 0으로
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)  # 겹친 넓이 ÷ 합친 넓이, 1e-9는 0 나눗셈 방지

# pred (N,5) [x1,y1,x2,y2,conf], gt (M,4) → TP·FP·FN 개수와 예측별 정답 여부
def match(pred, gt, iou_thr=0.5):
    """신뢰도 높은 예측부터 정답과 1:1로 짝짓기 → (TP, FP, FN, 맞은 예측 여부)"""
    order = np.argsort(-pred[:, 4]) if len(pred) else []  # 신뢰도 높은 순서의 인덱스
    ious = box_iou(pred[:, :4], gt) if len(pred) else np.zeros((0, len(gt)))
    used, hit = set(), np.zeros(len(pred), bool)  # 이미 짝지은 정답 번호, 예측별 맞음 여부
    for i in order:
        if len(gt) == 0:
            break
        j = int(np.argmax(np.where([k in used for k in range(len(gt))], -1, ious[i])))  # 쓴 정답은 -1로 가리고 IoU 최대인 정답
        if ious[i, j] >= iou_thr and j not in used:  # IoU 0.5 이상이면 TP (정답 1개당 1번만)
            used.add(j); hit[i] = True
    tp = int(hit.sum())
    return tp, len(pred) - tp, len(gt) - tp, hit  # FP = 예측 - TP, FN = 정답 - TP

# 예측은 낮은 임계값(0.05)으로 한 번만 수행해 저장
cache = []
for p in VAL_IMAGES:
    r = best_model(str(p), conf=0.05, imgsz=CFG['imgsz'], verbose=False)[0]
    pred = np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])  # 박스 4열 + 신뢰도 1열 = (N,5)
    cache.append((p, pred, read_gt(p)[0]))  # (경로, 예측, 정답 박스) 저장 — 모델 재실행 불필요

# 임계값 6개마다 전체 TP·FP·FN을 합산해 precision·recall·F1 계산
rows = []
for thr in [0.1, 0.25, 0.4, 0.55, 0.7, 0.85]:
    tp = fp = fn = 0
    for _, pred, gt in cache:
        t, f, n, _ = match(pred[pred[:, 4] >= thr], gt)  # 신뢰도 thr 이상인 예측만 남겨 매칭
        tp += t; fp += f; fn += n
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)  # max(..., 1): 분모 0 방지
    rows.append({'conf': thr, 'tp': tp, 'fp': fp, 'fn': fn, 'precision': round(prec, 3),
                 'recall': round(rec, 3), 'f1': round(2 * prec * rec / max(prec + rec, 1e-9), 3)})
thr_table = pd.DataFrame(rows)
BEST_CONF = float(thr_table.loc[thr_table['f1'].idxmax(), 'conf'])  # F1이 가장 큰 줄의 conf 값
print('F1이 가장 높은 임계값:', BEST_CONF)
thr_table

In [ ]:
# 임계값(x축)에 따른 precision·recall·F1 변화
plt.figure(figsize=(7, 3.5))
plt.plot(thr_table['conf'], thr_table['precision'], '-o', label='precision')
plt.plot(thr_table['conf'], thr_table['recall'], '-o', label='recall')
plt.plot(thr_table['conf'], thr_table['f1'], '-o', label='f1')
plt.axvline(BEST_CONF, ls='--', c='gray')  # 선택한 임계값 위치에 점선
plt.xlabel('confidence threshold'); plt.ylim(0, 1.05); plt.grid(alpha=.3); plt.legend(); plt.show()  # 세로축 0~1 고정

임계값을 올리면 FP(헛탐지)가 줄어 정밀도가 오르지만, FN(놓침)이 늘어 재현율이 떨어집니다.
**카운팅 서비스**라면 상자를 놓치는 것도, 없는 상자를 세는 것도 모두 숫자를 틀리게 만들므로 둘의 균형인 F1(정밀도와 재현율의 조화평균 — 둘 다 높아야 커지는 점수)이 가장 높은 값을 출발점으로 삼습니다.

### 2-6. 오류 분석 — 어디서 틀리는가

틀린 개수(FP + FN)가 가장 많은 이미지를 골라 직접 봅니다. 초록 = 정답, 파랑 = 맞은 예측, 빨강 = 틀린 예측(FP)입니다.

In [ ]:
# 이미지마다 BEST_CONF 기준 오류 수(FP + FN)를 세어 많은 순으로 정렬
err_rows = []
for p, pred, gt in cache:
    pr = pred[pred[:, 4] >= BEST_CONF]  # 선택한 임계값 이상인 예측만
    tp, fp, fn, hit = match(pr, gt)
    err_rows.append((fp + fn, fp, fn, p, pr, gt, hit))
err_rows.sort(key=lambda x: -x[0])  # 오류 개수 내림차순

# 오류가 가장 많은 4장: 초록 = 정답, 파랑 = 맞은 예측, 빨강 = FP
imgs, titles = [], []
for err, fp, fn, p, pr, gt, hit in err_rows[:4]:
    im = cv2.imread(str(p))
    draw_boxes(im, gt, (0, 255, 0), thick=3)
    draw_boxes(im, pr[hit], (255, 128, 0))  # hit=True인 예측 (BGR 파랑)
    draw_boxes(im, pr[~hit], (0, 0, 255), 'FP')  # ~hit: 맞지 않은 예측 = FP (빨강)
    imgs.append(im); titles.append(f'FP {fp} / FN {fn} | {p.name[:28]}')
show_grid(imgs, titles)
print('오류가 하나도 없는 이미지 비율:', round(np.mean([e[0] == 0 for e in err_rows]), 3))  # True 비율 = 평균

**확인해 보기** — 틀린 이미지들에서 공통점을 찾아 아래처럼 분류해 보세요. 개선 방향이 원인마다 다릅니다.

| 원인 유형 | 예시 | 개선 방향 |
|---|---|---|
| 겹침·밀집 | 상자가 겹쳐 하나로 잡힘 | 해상도(imgsz, 모델에 넣는 이미지 크기 — 픽셀) 올리기, 겹친 장면 데이터 추가 |
| 화면 가장자리 잘림 | 일부만 보이는 상자 | 라벨 기준 통일, 집계 영역에서 가장자리 제외 |
| 비슷한 물체 | 비닐 포대·작업자 손 | 헷갈리는 사례를 라벨링해 추가 |
| 라벨 누락 | 정답에 없는데 모델이 찾은 상자 | 라벨 검수 (모델이 맞는 경우도 있음) |

### 2-7. 개선 실험 (선택)

한 번에 하나만 바꿔서 비교해야 무엇이 효과가 있었는지 알 수 있습니다. `RUN_EXPERIMENT = True`로 바꾸면 두 가지 개선안을 추가로 학습해 기준 모델과 비교합니다 (T4 기준 약 40분 추가).

In [ ]:
RUN_EXPERIMENT = False  # True: 개선안 2개 추가 학습 (T4 기준 약 40분 추가)

# 한 번에 하나만 바꾼 개선안 — epoch 2배 / 모델 크기 n→s
EXPERIMENTS = [
    {'exp_name': 'more_epochs', 'model': 'yolo26n-seg.pt', 'epochs': CFG['epochs'] * 2, 'imgsz': CFG['imgsz']},
    {'exp_name': 'bigger_model', 'model': 'yolo26s-seg.pt', 'epochs': CFG['epochs'], 'imgsz': CFG['imgsz']},
]
METRIC_KEYS = ['box_map50', 'box_map50_95', 'mask_map50', 'mask_map50_95']  # 비교할 지표 이름

# 첫 줄 = 2-2에서 학습한 기준 모델 (설정 + val 지표 + 학습 시간)
exp_rows = [{'exp_name': 'baseline', 'model': 'yolo26n-seg.pt', 'epochs': CFG['epochs'], 'imgsz': CFG['imgsz'],
             **{k: val_row[k] for k in METRIC_KEYS}, 'train_min': round(TRAIN_MIN, 1)}]

if RUN_EXPERIMENT:
    for e in EXPERIMENTS:
        m = YOLO(e['model']); t0 = time.time()
        m.train(data='package-seg.yaml', epochs=e['epochs'], imgsz=e['imgsz'], batch=CFG['batch'],
                project='runs_box', name=e['exp_name'], exist_ok=True, seed=0, verbose=False)
        mins = (time.time() - t0) / 60  # 실험별 학습 시간(분)
        _, row = evaluate(YOLO(Path(m.trainer.save_dir) / 'weights' / 'best.pt'), 'val')  # 실험 모델의 최고 가중치로 val 평가
        exp_rows.append({**e, **{k: row[k] for k in METRIC_KEYS}, 'train_min': round(mins, 1)})  # 설정 + 지표를 한 줄로 합침

exp_table = pd.DataFrame(exp_rows)
exp_table.to_csv('experiments.csv', index=False); save_out('experiments.csv')  # CSV 저장 + Drive 복사
exp_table

개선안이 기준 모델보다 좋아졌더라도 **학습 시간·추론(inference, 학습된 모델로 예측하는 것) 속도**가 함께 늘어나는지 확인해야 합니다. 3회차에서 다룰 엣지 장비(현장에 두는 소형 AI 컴퓨터, 예: Jetson)에서는 작은 모델의 속도가 더 중요할 수 있습니다.

---
## Step 3. 트래킹 — 같은 상자에 같은 ID 붙이기

### 3-1. 실습용 컨베이어 영상 만들기

이 데이터셋에는 같은 CCTV에서 연속으로 뽑은 프레임(`frame_1` ~ `frame_307`)이 들어 있습니다. 이 프레임들을 순서대로 이어 붙여 약 17초짜리 컨베이어 영상을 만듭니다.
CCTV 화면은 **좌우 두 화면이 한 장에 붙어 있는 형태**라, 상자가 왼쪽에서 오른쪽으로 흐르는 **오른쪽 화면만 잘라** 씁니다.
학습 세트의 이미지에는 밝기·회전 등 변형(augmentation, 학습 데이터를 늘리려고 밝기·회전 등을 바꾼 것)을 준 사본이 3장씩 있으므로, 바로 앞 프레임과 가장 비슷한 사본을 골라 자연스럽게 잇습니다.

> 이 영상에는 학습에 쓰인 프레임이 섞여 있습니다. 그래서 여기서는 **정확도 평가가 아니라 트래킹·집계 기능을 구현하는 연습용**으로 씁니다. 정확도 평가는 Step 2의 val/test 결과를 기준으로 합니다.

In [ ]:
# build_conveyor_clip: frame_1~307 이미지를 이어 영상 생성 → (파일명, 프레임 수, fps, (폭, 높이))
def build_conveyor_clip(root, out='conveyor_clip.mp4', first=1, last=307, fps=15, panel='right'):
    by_num = defaultdict(list)  # 프레임 번호 → 같은 번호 파일들(원본+변형 사본)
    for f in glob.glob(str(root / 'images' / '*' / 'frame_*_jpg*')):
        n = int(re.search(r'frame_(\d+)_jpg', f).group(1))  # 파일 이름에서 프레임 번호 추출
        if first <= n <= last:
            by_num[n].append(f)
    nums = sorted(by_num)
    load = lambda f: cv2.imread(f).astype(np.float32)  # 픽셀 차이 계산용 실수형 읽기
    anchor = next(n for n in nums if '/train/' not in by_num[n][0])   # 변형이 없는 val/test 프레임에서 출발
    chosen = {anchor: by_num[anchor][0]}  # 프레임 번호 → 고른 파일
    a = nums.index(anchor)
    for seq in (nums[a + 1:], nums[:a][::-1]):                         # 앞쪽·뒤쪽으로 이어 가며 선택
        prev = load(chosen[anchor])
        for n in seq:
            diffs = [np.abs(load(c) - prev).mean() for c in by_num[n]]  # 직전 프레임과의 평균 픽셀 차이
            chosen[n] = by_num[n][int(np.argmin(diffs))]  # 차이가 가장 작은 사본 선택
            prev = load(chosen[n])
    crop = lambda im: im[:, im.shape[1] // 2:] if panel == 'right' else im[:, :im.shape[1] // 2]  # 폭 절반 기준 오른쪽/왼쪽 화면만
    h, w = crop(cv2.imread(chosen[nums[0]])).shape[:2]
    writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))  # mp4v 코덱, 초당 fps장
    for n in nums:
        writer.write(np.ascontiguousarray(crop(cv2.imread(chosen[n]))))  # 잘라 낸 배열을 메모리상 연속으로 만들어 기록
    writer.release()
    return out, len(nums), fps, (w, h)

CLIP, n_frames, CLIP_FPS, (W, H) = build_conveyor_clip(DATA_ROOT)
save_out(CLIP)  # Drive에도 복사 (통합 실습에서 사용)
print(f'{CLIP}: {n_frames}프레임, {CLIP_FPS}fps, {W}x{H}, 약 {n_frames / CLIP_FPS:.1f}초')
show_video(CLIP)

### 3-2. 탐지만으로는 개수를 셀 수 없는 이유

프레임마다 탐지된 상자 수를 모두 더하면 '실제로 지나간 상자 수'와 전혀 다른 숫자가 나옵니다. 같은 상자가 수십 프레임에 걸쳐 반복해서 잡히기 때문입니다.

In [ ]:
# stream=True: 프레임별 결과를 하나씩 꺼냄 (메모리 절약) / len(r.boxes): 그 프레임 탐지 수
per_frame = [len(r.boxes) for r in best_model(CLIP, stream=True, conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)]
print('프레임별 탐지 수 합계:', sum(per_frame), '| 프레임당 평균:', round(np.mean(per_frame), 2))

### 3-3. 트래킹 적용

트래커(tracker, 프레임 사이 같은 물체를 이어 주는 추적 알고리즘)는 이전 프레임의 박스 위치와 움직임을 기억해 두었다가, 새 프레임의 탐지 결과와 짝을 지어 **같은 물체에 같은 ID**를 붙입니다.
여기서는 빠르고 널리 쓰이는 **ByteTrack**(`bytetrack.yaml` — 프레임 사이 같은 물체에 같은 번호를 붙이는 추적 알고리즘, 신뢰도가 낮은 박스도 짝짓기에 활용)을 씁니다.

> 영상 파일 전체를 `track(source=영상, stream=True)`(`stream=True` = 결과를 프레임마다 하나씩 꺼내 쓰는 방식)로 한 번에 넘기면 트래커가 영상 시작부터 끝까지 상태를 알아서 이어 갑니다. 반대로 카메라 프레임을 한 장씩 꺼내 `track(frame, persist=True)`를 반복 호출할 때는 `persist=True`(트래커 상태를 다음 호출까지 유지)를 꼭 줘야 이전 프레임의 ID가 유지됩니다. 통합 실습에서 이 방식을 씁니다.

In [ ]:
# id_color: ID 번호 → 항상 같은 BGR 색 (ID를 난수 시드로 사용)
def id_color(tid):
    rng = np.random.default_rng(int(tid))
    return tuple(int(c) for c in rng.integers(60, 255, 3))

# run_tracking: 영상 트래킹 → 결과 영상 저장, ID별 중심점 이력 반환 (trail = 궤적 길이)
def run_tracking(src, dst='tracked.mp4', tracker='bytetrack.yaml', trail=25):
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    history = defaultdict(list)      # track_id → [(frame, cx, cy), ...]
    # 영상 전체를 넘기면 트래커가 처음~끝까지 ID 상태를 이어 감
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        frame = r.orig_img.copy()  # 원본 프레임 사본 위에 그림
        if r.boxes.id is not None:  # ID가 아직 안 붙은 프레임은 id가 None
            # 박스 좌표와 트래킹 ID를 짝지어 하나씩 (GPU 텐서 → 파이썬 값)
            for (x1, y1, x2, y2), tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist()):
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2  # 박스 중심 좌표 (픽셀)
                history[tid].append((i, cx, cy))
                col = id_color(tid)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
                cv2.putText(frame, f'ID {tid}', (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
                pts = np.array([(int(x), int(y)) for _, x, y in history[tid][-trail:]], np.int32)  # 최근 trail개 중심점
                cv2.polylines(frame, [pts], False, col, 2)  # 중심점을 이어 이동 궤적 그리기
        writer.write(frame)
    writer.release()
    return history

track_hist = run_tracking(CLIP)
# ID별 등장 프레임 수, 처음·마지막 프레임, 이동량(dx, dy) 표
life = pd.DataFrame([{'track_id': t, 'frames': len(h), 'first_frame': h[0][0], 'last_frame': h[-1][0],
                      'dx': round(float(h[-1][1] - h[0][1]), 1), 'dy': round(float(h[-1][2] - h[0][2]), 1)}
                     for t, h in track_hist.items()]).sort_values('first_frame')
print('부여된 ID 수:', len(life), '| 5프레임 이상 이어진 ID 수:', int((life['frames'] >= 5).sum()))
show_video('tracked.mp4')
life.head(10)  # 첫 등장 순 앞 10개 ID

**확인해 보기**
- `dx`(처음 위치 대비 마지막 위치의 x 이동량)의 부호를 보면 상자가 어느 방향으로 움직이는지 알 수 있습니다.
- 몇 프레임만 살다 사라진 짧은 ID들은 무엇일까요? 상자가 가려졌다 다시 보이며 **새 ID를 받는 현상(ID switch)** 은 집계 오차의 주요 원인입니다.

---
## Step 4. 라인 카운팅

화면에 **가상의 기준선**을 긋고, 어떤 ID의 중심점이 이전 프레임에는 선의 한쪽에, 현재 프레임에는 반대쪽에 있으면 '통과'로 셉니다. ID마다 한 번만 세므로 같은 상자를 중복해서 세지 않습니다.

```
트래킹 ID별 중심점 이력 → 선을 넘었는가? → 처음 넘은 ID만 카운트 → 화면 표시 + 로그 기록
```

### 4-1. 이동 방향과 기준선 위치 정하기

In [ ]:
moving = life[(life['frames'] >= 5) & (life['dx'].abs() > W * 0.1)]   # 충분히 오래, 충분히 멀리 움직인 ID만
main_dir = 'right_to_left' if moving['dx'].median() < 0 else 'left_to_right'  # dx 중앙값 음수 = 오른쪽→왼쪽
print('상자의 주 이동 방향:', main_dir, '| dx 중앙값:', moving['dx'].median())

# 0.5 → 0.3이면 선이 왼쪽으로 이동, Step 4·5 카운트와 size_rules.json에 반영
LINE_X = int(W * 0.5)    # 화면 가운데에 세로 기준선 (필요하면 비율을 바꿔 보세요)
cap = cv2.VideoCapture(CLIP); cap.set(cv2.CAP_PROP_POS_FRAMES, 100); _, preview = cap.read(); cap.release()  # 100번째 프레임 1장
cv2.line(preview, (LINE_X, 0), (LINE_X, H), (0, 255, 255), 3)  # 노란 세로선 (BGR)
show_bgr(preview, f'Counting line at x={LINE_X}', size=(4, 7))

In [ ]:
# run_counting: 영상 트래킹 + 기준선 통과 카운트 → 결과 영상 저장, 통과 기록 DataFrame 반환
def run_counting(src, dst='counting.mp4', line_x=LINE_X, tracker='bytetrack.yaml'):
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    last_x, counted, log = {}, {}, []  # ID별 직전 중심 x, 센 ID→방향, 통과 기록
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        frame = r.orig_img.copy()
        if r.boxes.id is not None:
            for (x1, y1, x2, y2), tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist()):
                cx = (x1 + x2) / 2  # 박스 중심 x좌표 (픽셀)
                prev = last_x.get(tid)
                if prev is not None and tid not in counted:  # 직전 위치가 있고 아직 안 센 ID만 → ID당 1번
                    if prev < line_x <= cx:  # 직전엔 선 왼쪽, 지금은 선 위/오른쪽
                        counted[tid] = 'left_to_right'
                    elif prev >= line_x > cx:  # 직전엔 선 위/오른쪽, 지금은 선 왼쪽
                        counted[tid] = 'right_to_left'
                    if tid in counted:  # 이번 프레임에 방금 셌으면 기록
                        log.append({'frame': i, 'time_s': round(i / CLIP_FPS, 2), 'track_id': tid,
                                    'direction': counted[tid]})
                last_x[tid] = cx  # 다음 프레임 비교용으로 현재 위치 저장
                col = (0, 200, 0) if tid in counted else (0, 165, 255)  # 센 상자 초록, 아직 안 센 상자 주황
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
                cv2.putText(frame, f'ID {tid}', (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        flash = log and log[-1]['frame'] >= i - 2          # 방금 카운트되면 선을 잠깐 빨갛게
        cv2.line(frame, (line_x, 0), (line_x, H), (0, 0, 255) if flash else (0, 255, 255), 3)
        # 왼쪽 위 검은 상자(-1 = 속 채우기)에 누적 카운트 표시
        cv2.rectangle(frame, (8, 8), (200, 48), (0, 0, 0), -1)
        cv2.putText(frame, f'COUNT: {len(counted)}', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        writer.write(frame)
    writer.release()
    return pd.DataFrame(log, columns=['frame', 'time_s', 'track_id', 'direction'])

count_log = run_counting(CLIP)
count_log.to_csv('box_count_log.csv', index=False); save_out('box_count_log.csv')  # CSV 저장 + Drive 복사
print('통과한 상자 수:', len(count_log))
show_video('counting.mp4')
count_log

**확인해 보기** — `counting.mp4`를 보면서 선을 넘는 상자를 직접 세어 보세요. 모델의 카운트와 다르다면 어느 장면에서 차이가 났나요? 상자가 붙어서 하나로 잡혔는지, 선 근처에서 ID가 바뀌었는지 확인합니다.

여기까지가 '몇 개 지나갔나'입니다. 현장에서는 여기서 한 걸음 더 나아가 **어떤 크기의 상자가 몇 개 지나갔나**를 알아야 분류 라인과 적재 공간을 계획할 수 있습니다.

---
## Step 5. 크기 분류 — 크기별로 세기

```
마스크 → 크기 측정 (면적, 긴 변, 짧은 변) → S/M/L 기준을 데이터로 정하기
      → 트랙별 크기 확정 (가장자리에서 잘린 프레임 제외, 중앙값) → 크기별 카운트 + 영상 + 로그
```

### 5-1. 마스크로 크기 재기

마스크 윤곽선(`r.masks.xy`)에서 세 가지 값을 구합니다.

| 값 | 계산 | 의미 |
|---|---|---|
| `area_px` | 윤곽선 내부 면적 `cv2.contourArea` | 화면에서 상자가 차지하는 실제 면적 |
| `long_px`, `short_px` | 윤곽선을 가장 작게 감싸는 **회전 사각형** `cv2.minAreaRect`의 두 변 | 놓인 각도와 상관없는 상자의 가로·세로 |

In [ ]:
# measure_mask: 윤곽선 (N,2) → 면적, 긴 변, 짧은 변, 회전 사각형 꼭짓점 4개
def measure_mask(poly):
    """마스크 윤곽선 (N,2) → (면적, 긴 변, 짧은 변, 회전 사각형 꼭짓점)"""
    pts = np.asarray(poly, np.float32)
    area = float(cv2.contourArea(pts))  # 윤곽선 내부 면적 (px²)
    rect = cv2.minAreaRect(pts)  # 가장 작은 회전 사각형 ((중심), (폭, 높이), 각도)
    (w_, h_) = rect[1]  # 회전 사각형의 폭·높이
    return area, float(max(w_, h_)), float(min(w_, h_)), cv2.boxPoints(rect)  # 큰 값=긴 변, 작은 값=짧은 변

# 영상의 한 프레임에서 확인
cap = cv2.VideoCapture(CLIP); cap.set(cv2.CAP_PROP_POS_FRAMES, 100); _, frame = cap.read(); cap.release()
r = best_model(frame, conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)[0]  # 프레임 1장 추론
vis = frame.copy()
# 탐지가 없으면 masks가 None → 빈 목록으로 대신
for b, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy if r.masks is not None else []):
    if len(poly) < 3:
        continue
    area, long_px, short_px, corners = measure_mask(poly)
    vis = draw_polys(vis, [poly], (0, 255, 0))
    cv2.polylines(vis, [corners.astype(np.int32)], True, (255, 0, 255), 2)  # 회전 사각형 (자홍색)
    draw_boxes(vis, [b], (0, 0, 255), thick=1)
    # 박스 위에 면적과 '긴 변x짧은 변' 표시
    cv2.putText(vis, f'{area:.0f}px2', (int(b[0]), int(b[1]) - 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    cv2.putText(vis, f'{long_px:.0f}x{short_px:.0f}', (int(b[0]), int(b[1]) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
show_bgr(vis, 'green = mask, magenta = rotated rect, red = box', size=(4, 7))

### 5-2. S/M/L 기준을 데이터로 정하기

기준값을 감으로 정하지 않고 **정답 라벨에서** 정합니다. 같은 카메라(`frame_*` 이미지)의 오른쪽 화면에 있는 상자 다각형 면적을 모두 모아 분포를 보고, 하위 1/3과 상위 1/3을 가르는 값(33%, 67% 분위수(percentile — 크기순으로 세웠을 때 해당 비율 위치의 값))을 S/M/L 경계로 씁니다.

- 학습용 사본에는 회전·확대 변형이 섞여 있어 크기가 달라질 수 있으므로 **변형이 없는 val/test 라벨만** 씁니다.
- 화면 가장자리에 걸려 **일부만 보이는 상자는 제외**합니다. 잘린 상자는 실제보다 작게 재지기 때문입니다.

In [ ]:
# 키우면 가장자리 판정이 넓어져 제외되는 상자가 늘어남
EDGE = 4   # 이 픽셀 안쪽으로 가장자리에 닿으면 '잘린 상자'로 봅니다

label_rows = []
for split in ['val', 'test']:  # 변형이 없는 val/test 라벨만 (train 사본은 회전·확대 변형)
    for img_path in (DATA_ROOT / 'images' / split).glob('frame_*.jpg'):  # 같은 카메라 프레임만
        _, polys = read_gt(img_path)
        for poly in polys:
            xs, ys = poly[:, 0] - 320, poly[:, 1]          # 오른쪽 화면 기준 좌표 (영상과 같은 좌표계)
            if xs.min() < EDGE or xs.max() > W - EDGE or ys.min() < EDGE or ys.max() > H - EDGE:
                continue                                    # 왼쪽 화면 상자이거나 가장자리에 잘린 상자
            area, long_px, short_px, _ = measure_mask(np.stack([xs, ys], 1))
            label_rows.append({'image': img_path.name, 'area_px': area, 'long_px': long_px, 'short_px': short_px})
label_sizes = pd.DataFrame(label_rows)  # 라벨 상자별 면적·긴 변·짧은 변 표

q33, q67 = np.percentile(label_sizes['area_px'], [33.3, 66.7])  # 하위 1/3, 상위 1/3 경계값
# S 상한, M 상한과 측정 조건을 한 딕셔너리로 → 통합 실습도 같은 기준 사용
SIZE_RULES = {'measure': 'mask_area_px', 'small_max': round(float(q33)), 'medium_max': round(float(q67)),
              'edge_margin_px': EDGE, 'line_x': LINE_X, 'conf': BEST_CONF, 'imgsz': CFG['imgsz']}
json.dump(SIZE_RULES, open('size_rules.json', 'w'), indent=2); save_out('size_rules.json')  # JSON 파일로 저장 + Drive 복사
print(f'라벨 상자 {len(label_sizes)}개로 기준을 정했습니다:', SIZE_RULES)

# 면적 분포 히스토그램과 S|M, M|L 경계선
plt.figure(figsize=(8, 3.2))
plt.hist(label_sizes['area_px'], bins=40, color='tab:gray')
for v, name in [(q33, 'S | M'), (q67, 'M | L')]:
    plt.axvline(v, c='red', ls='--'); plt.text(v, plt.ylim()[1] * 0.9, f' {name}\n {v:.0f}', color='red')  # 경계선 + 위쪽에 값 표시
plt.xlabel('mask area (px^2)'); plt.ylabel('labels'); plt.title('Package size distribution (ground truth)')
plt.show()

In [ ]:
# size_class: 면적 → 'S'(small_max 이하) / 'M'(medium_max 이하) / 'L'
def size_class(area, rules=SIZE_RULES):
    if area <= rules['small_max']:
        return 'S'
    if area <= rules['medium_max']:
        return 'M'
    return 'L'

SIZE_COLOR = {'S': (255, 200, 0), 'M': (0, 200, 255), 'L': (60, 60, 255)}   # BGR: 하늘색, 주황, 빨강
print([size_class(a) for a in [1000, 5000, 15000]])  # 경계 아래·사이·위 값으로 동작 확인

### 5-3. 관측 기록 만들기 — 한 번 추적하고 표로 남기기

트래킹을 한 번만 돌리면서 **프레임마다, 상자마다** 위치와 크기를 표 한 줄로 기록합니다. 이후의 집계·분류·영상 만들기는 모두 이 표로 합니다.
모델을 다시 돌리지 않고도 기준을 바꿔 가며 몇 번이고 다시 집계할 수 있습니다.

In [ ]:
# collect_observations: 영상 1회 트래킹 → (관측 표, (프레임, ID)→윤곽선 딕셔너리)
def collect_observations(src, tracker='bytetrack.yaml'):
    obs, polys = [], {}
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        if r.boxes.id is None or r.masks is None:  # ID나 마스크가 없는 프레임은 건너뜀
            continue
        for b, tid, poly in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(), r.masks.xy):  # 박스·ID·윤곽선을 같은 순서로
            if len(poly) < 3:
                continue
            area, long_px, short_px, _ = measure_mask(poly)  # 이 관측의 크기
            x1, y1, x2, y2 = b
            # 관측 1줄 — at_edge: 박스가 화면 가장자리에 닿았는지 (크기 결정에서 제외)
            obs.append({'frame': i, 'track_id': tid, 'cx': (x1 + x2) / 2, 'cy': (y1 + y2) / 2,
                        'area_px': area, 'long_px': long_px, 'short_px': short_px,
                        'at_edge': bool(x1 < EDGE or y1 < EDGE or x2 > W - EDGE or y2 > H - EDGE)})
            polys[(i, tid)] = poly  # 5-5 영상 다시 그리기용
    return pd.DataFrame(obs), polys

obs, obs_polys = collect_observations(CLIP)
print('관측 행 수:', len(obs), '| 트랙 수:', obs['track_id'].nunique(),
      '| 가장자리에 걸린 관측 비율:', round(obs['at_edge'].mean(), 2))
obs.round(1).head()  # 소수점 1자리, 앞 5줄

In [ ]:
# 한 상자의 면적이 시간에 따라 어떻게 재지는지 확인
tid_example = obs.groupby('track_id').size().idxmax()  # groupby: ID별로 묶어 관측 수 → 가장 많은 ID
one = obs[obs['track_id'] == tid_example]  # 그 ID의 관측만
plt.figure(figsize=(8, 3))
plt.plot(one['frame'], one['area_px'], '-', c='gray', lw=1)
plt.scatter(one['frame'], one['area_px'], c=np.where(one['at_edge'], 'red', 'tab:blue'), s=12)  # 가장자리=빨강, 나머지=파랑
plt.xlabel('frame'); plt.ylabel('mask area (px^2)')
plt.title(f'track {tid_example}: red = touching frame edge (excluded)')
plt.grid(alpha=.3); plt.show()

상자가 화면에 들어오거나 나갈 때(빨간 점)는 일부만 보여 면적이 작게 재집니다. 그래서 **가장자리에 걸리지 않은 관측만 모아 중앙값**(median, 크기순으로 세웠을 때 가운데 값)으로 그 상자의 크기를 정합니다. 평균 대신 중앙값을 쓰면 마스크가 순간적으로 튀는 값에도 흔들리지 않습니다.

### 5-4. 크기별로 세기

In [ ]:
# summarize_tracks: 관측 표 → 기준선을 넘은 트랙별 통과 시점·방향·크기 표 (모델 재실행 없음)
def summarize_tracks(obs, line_x=LINE_X, fps=CLIP_FPS, rules=SIZE_RULES):
    rows = []
    for tid, g in obs.sort_values('frame').groupby('track_id'):  # 프레임 순 정렬 후 ID별로 묶음 (g = 한 ID의 관측)
        prev = g['cx'].shift()  # 바로 앞 관측의 중심 x를 같은 줄에
        # 앞·현재 관측이 선의 양쪽이면 통과 (&가 |보다 먼저 계산)
        crossed = g[(prev < line_x) & (g['cx'] >= line_x) | (prev >= line_x) & (g['cx'] < line_x)]
        if crossed.empty:
            continue                                     # 기준선을 넘지 않은 트랙은 집계하지 않음
        first = crossed.iloc[0]  # 처음 넘은 관측
        direction = 'left_to_right' if first['cx'] >= line_x else 'right_to_left'
        full = g[~g['at_edge']]  # 가장자리에 걸리지 않은 관측만
        use = full if len(full) >= 3 else g             # 온전한 관측이 3개 미만이면 전체 사용
        area = float(use['area_px'].median())  # 중앙값 — 순간적으로 튀는 값에 강함
        rows.append({'track_id': tid, 'cross_frame': int(first['frame']),
                     'time_s': round(first['frame'] / fps, 2), 'direction': direction,
                     'size_class': size_class(area, rules), 'area_px': round(area),
                     'long_px': round(float(use['long_px'].median())),
                     'short_px': round(float(use['short_px'].median())),
                     'n_obs': len(use), 'edge_only': len(full) < 3})
    cols = ['track_id', 'cross_frame', 'time_s', 'direction', 'size_class', 'area_px',
            'long_px', 'short_px', 'n_obs', 'edge_only']
    return pd.DataFrame(rows, columns=cols).sort_values('cross_frame').reset_index(drop=True)

size_log = summarize_tracks(obs)
size_log.to_csv('box_size_log.csv', index=False); save_out('box_size_log.csv')

size_counts = size_log['size_class'].value_counts().reindex(['S', 'M', 'L'], fill_value=0)  # S/M/L 순서, 없는 등급은 0
print('크기별 통과 수:', size_counts.to_dict(), '| 합계:', int(size_counts.sum()))
print('Step 4 카운트와 같은가:', int(size_counts.sum()) == len(count_log))
size_log

Step 4의 카운트와 합계가 같아야 합니다. 같은 모델·같은 트래커·같은 기준선을 썼기 때문입니다. 다르다면 `collect_observations()`에서 마스크가 없는 상자를 건너뛴 경우인지 확인해 보세요.

### 5-5. 결과 영상과 요약 그래프

관측 표와 집계 결과를 이용해 영상을 다시 그립니다. 상자는 크기별 색으로 칠하고, 기준선을 넘는 순간 오른쪽 위의 S/M/L 카운터가 올라갑니다.

In [ ]:
# render_size_video: 저장된 윤곽선·집계로 영상을 다시 그려 dst에 저장, 파일명 반환 (모델 재실행 없음)
def render_size_video(src, obs_polys, size_log, dst='size_counting.mp4', line_x=LINE_X):
    cls_of = dict(zip(size_log['track_id'], size_log['size_class']))  # ID → 크기 등급
    cross_at = dict(zip(size_log['track_id'], size_log['cross_frame']))  # ID → 기준선 통과 프레임
    by_frame = defaultdict(list)  # 프레임 번호 → [(ID, 윤곽선), ...]
    for (i, tid), poly in obs_polys.items():
        by_frame[i].append((tid, poly))
    cap = cv2.VideoCapture(src)
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    i = 0
    while True:
        ok, frame = cap.read()  # ok=False면 영상 끝
        if not ok:
            break
        overlay = frame.copy()
        for tid, poly in by_frame[i]:
            c = cls_of.get(tid)
            col = SIZE_COLOR[c] if c else (180, 180, 180)          # 집계 대상이 아닌 상자는 회색
            cv2.fillPoly(overlay, [poly.astype(np.int32)], col)
            x, y = poly[:, 0].min(), poly[:, 1].min()
            label = f'{c} #{tid}' if c else f'#{tid}'
            cv2.putText(frame, label, (int(x), int(max(14, y - 4))), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        frame = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0)  # 칠한 사본 45% + 원본 55% → 반투명
        cv2.line(frame, (line_x, 0), (line_x, H), (0, 255, 255), 2)
        done = [cls_of[t] for t, f in cross_at.items() if f <= i]  # 현재 프레임까지 선을 넘은 상자들의 등급
        # 위쪽 검은 띠에 'S:n M:n L:n =합계' 표시
        cv2.rectangle(frame, (6, 6), (W - 6, 40), (0, 0, 0), -1)
        for k, s in enumerate(['S', 'M', 'L']):
            cv2.putText(frame, f'{s}:{done.count(s)}', (14 + k * 72, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.75, SIZE_COLOR[s], 2)
        cv2.putText(frame, f'={len(done)}', (14 + 3 * 72, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
        writer.write(frame)
        i += 1
    cap.release(); writer.release()
    return dst

render_size_video(CLIP, obs_polys, size_log)
save_out('size_counting.mp4')  # Drive에도 복사
show_video('size_counting.mp4')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [1, 2]})
bar_colors = ['#00c8ff', '#ffc800', '#ff3c3c']  # SIZE_COLOR와 같은 색 (matplotlib은 RGB 순서)
axes[0].bar(['S', 'M', 'L'], size_counts.values, color=bar_colors)
axes[0].set_title('Packages by size'); axes[0].set_ylabel('count')
for s, c in zip(['S', 'M', 'L'], bar_colors):
    d = size_log[size_log['size_class'] == s]
    axes[1].scatter(d['time_s'], d['area_px'], s=60, color=c, label=s, edgecolor='k')  # 상자별 통과 시각(x)과 면적(y)
axes[1].axhline(SIZE_RULES['small_max'], ls='--', c='gray', lw=1)  # S|M 경계
axes[1].axhline(SIZE_RULES['medium_max'], ls='--', c='gray', lw=1)  # M|L 경계
axes[1].set_xlabel('time crossing the line (s)'); axes[1].set_ylabel('mask area (px^2)')
axes[1].set_title('Each counted package'); axes[1].legend(); axes[1].grid(alpha=.3)
# show() 뒤에는 그림이 비워지므로 savefig를 먼저
plt.tight_layout(); plt.savefig('size_summary.png', dpi=120); save_out('size_summary.png'); plt.show()

결과 파일 (모두 `SAVE_DIR`에도 저장됩니다)

| 파일 | 내용 |
|---|---|
| `box_seg_best.pt` | 파인튜닝한 상자 세그멘테이션 모델 |
| `conveyor_clip.mp4` | 실습용 컨베이어 영상 |
| `box_count_log.csv` | 통과 기록 (`frame, time_s, track_id, direction`) |
| `size_rules.json` | S/M/L 기준값, 기준선 위치, 임계값 |
| `box_size_log.csv` | 크기별 통과 기록 (`track_id, cross_frame, time_s, direction, size_class, area_px, long_px, short_px, n_obs, edge_only`) |
| `size_counting.mp4` | 크기별 색과 S/M/L 카운터가 표시된 영상 |
| `size_summary.png` | 크기별 개수와 상자별 면적 그래프 |
| `experiments.csv` | 실험별 성능 비교 |

`box_seg_best.pt`, `conveyor_clip.mp4`, `size_rules.json`은 통합 실습에서 다시 사용합니다.

---
## 도전 과제

1. **긴 변 기준으로 분류** — 면적 대신 `long_px`(긴 변)로 S/M/L을 나누면 결과가 어떻게 달라지나요? 납작하고 긴 상자는 어느 쪽 기준에서 더 합리적으로 분류되나요?
2. **원근 보정** — 카메라에서 먼 쪽(화면 위쪽)의 상자는 작게 보입니다. 기준선 위치(세로)에 따라 면적을 보정하는 방법을 생각해 보세요. 힌트: 라벨 면적과 `cy`(세로 위치)의 관계를 그래프로 그려 봅니다.
3. **실시간으로 결정하기** — 지금은 영상을 다 본 뒤 크기를 정합니다. 기준선을 넘는 **그 순간까지** 모인 관측만으로 크기를 정하도록 `summarize_tracks()`를 바꿔 보세요. 결과가 얼마나 달라지나요?
4. **트래커 비교** — `tracker='botsort.yaml'`(BoT-SORT — ByteTrack에 카메라 움직임 보정 등을 더한 트래커)로 바꿔 ID 수와 크기별 카운트가 어떻게 달라지는지 표로 정리해 보세요.

## 정리

- 사전학습 모델은 출발점일 뿐, **우리 대상은 우리 데이터로 가르쳐야** 합니다.
- 크기를 재려면 박스가 아니라 **마스크**가 필요합니다. 비스듬한 상자의 박스는 실제보다 훨씬 큽니다.
- 기준값은 감이 아니라 **정답 데이터의 분포**로 정하고, 파일(`size_rules.json`)로 남겨야 다른 시스템에서도 같은 기준을 씁니다.
- 영상 분석은 '추적해서 **관측 표로 남기고** → 표로 집계'하는 구조로 만들면, 기준을 바꿔도 모델을 다시 돌릴 필요가 없습니다.

## 용어 정리

| 용어 | 뜻 |
|---|---|
| 사전학습(pretrained) | COCO 같은 공개 대규모 데이터로 미리 학습해 둔 상태입니다. `yolo26n.pt`, `yolo26n-seg.pt`가 사전학습 가중치입니다. |
| 파인튜닝(fine-tuning) | 사전학습된 가중치에서 출발해 우리 데이터로 추가 학습하는 것입니다. 처음부터 학습하는 것보다 적은 데이터와 시간으로 좋은 성능을 냅니다. |
| 가중치(weights, `.pt`) | 모델이 학습으로 얻은 숫자 값들을 담은 파일입니다. `best.pt`는 검증 점수가 가장 좋았던 epoch의 가중치입니다. |
| 클래스(class) | 모델이 구분하는 물체 종류입니다. 이 데이터셋은 `package` 1개, COCO는 80개입니다. |
| 라벨(label) | 사람이 이미지에 표시해 둔 정답입니다. 이 데이터셋은 상자 테두리를 0~1로 정규화된 다각형 꼭짓점 좌표로 적어 둡니다. |
| 탐지(detect) | 물체의 위치를 사각형 박스로 찾는 방식입니다. |
| 세그멘테이션 마스크 | 물체가 차지하는 픽셀 영역을 모양 그대로 나타낸 것입니다. 이 노트북에서는 `r.masks.xy`의 윤곽선 좌표로 다룹니다. |
| epoch | 학습 데이터 전체를 한 번 모두 학습하는 단위입니다. |
| batch | 한 번에 묶어서 모델에 넣는 이미지 장수입니다. 크면 빠르지만 GPU 메모리를 많이 씁니다. |
| imgsz | 모델에 넣을 때 맞추는 이미지 크기(픽셀)입니다. 키우면 작은 물체를 잘 찾지만 느려집니다. |
| loss(손실) | 예측이 정답과 얼마나 다른지 나타내는 값입니다. 작을수록 좋으며, 세그멘테이션 모델은 마스크 손실(seg_loss)이 더 있습니다. |
| train / val / test 세트 | 학습용 / 학습 중 성능 확인용(검증) / 마지막 성능 확인용(테스트) 데이터입니다. val·test는 학습에 쓰지 않습니다. |
| 과적합(overfitting) | 학습 데이터만 외워서 train loss는 내려가는데 val loss는 올라가는 상태입니다. |
| 추론(inference) | 학습된 모델로 새 이미지·영상에 대해 예측하는 것입니다. |
| IoU | Intersection over Union. 예측과 정답이 겹치는 넓이를 둘을 합친 넓이로 나눈 값입니다. 1이면 완전히 일치합니다. |
| TP / FP / FN | TP는 맞게 찾은 상자, FP는 없는 상자를 찾은 헛탐지, FN은 있는데 놓친 상자입니다. |
| Precision / Recall | Precision(정밀도)은 상자라고 한 것 중 진짜 상자의 비율, Recall(재현율)은 진짜 상자 중 찾아낸 비율입니다. |
| F1 | Precision과 Recall의 조화평균입니다. 둘 중 하나만 높아서는 커지지 않으므로 균형을 보는 지표입니다. |
| mAP50 / mAP50-95 | mAP50은 IoU 0.5 기준 정밀도-재현율 곡선 아래 면적, mAP50-95는 IoU 0.5~0.95 구간의 평균입니다. 뒤쪽이 위치·모양 정확도를 더 엄격하게 봅니다. (B)는 박스, (M)은 마스크 기준입니다. |
| 신뢰도 임계값(conf) | 이 값보다 신뢰도가 낮은 예측은 버리는 기준입니다. 올리면 FP가 줄고 FN이 늘어납니다. |
| 일반화 | 학습에 쓰지 않은 새 데이터에서도 비슷한 성능을 내는 것입니다. |
| augmentation(데이터 변형) | 학습 데이터를 늘리려고 밝기·회전·확대 등을 바꾼 사본을 만드는 것입니다. |
| 프레임(frame) / fps | 영상을 이루는 낱장 이미지입니다. fps는 1초에 보여 주는 프레임 수입니다. |
| 트래킹 / ByteTrack | 프레임이 바뀌어도 같은 물체에 같은 ID를 붙이는 기술입니다. ByteTrack은 빠르고 널리 쓰이는 트래커이며 `bytetrack.yaml`로 지정합니다. |
| BoT-SORT | ByteTrack에 카메라 움직임 보정 등을 더한 트래커입니다. `botsort.yaml`로 지정합니다. |
| stream=True | 영상 결과를 한꺼번에 모으지 않고 프레임마다 하나씩 꺼내 쓰는 방식입니다. 메모리를 아낍니다. |
| persist=True | 프레임을 한 장씩 `track()`에 넣을 때 트래커 상태를 다음 호출까지 유지해 ID가 이어지게 하는 옵션입니다. |
| ID switch | 같은 물체가 가려졌다 다시 보이는 등의 이유로 새 ID를 받는 현상입니다. 중복 카운트나 누락의 원인이 됩니다. |
| 라인 카운팅 | 가상의 기준선을 정하고, 트래킹 ID의 중심점이 선을 넘을 때 ID당 한 번씩 세는 방법입니다. |
| 회전 사각형(minAreaRect) | 윤곽선을 가장 작게 감싸도록 기울어질 수 있는 사각형입니다. 놓인 각도와 상관없이 물체의 가로·세로를 잴 수 있습니다. |
| 분위수(percentile) | 값을 크기순으로 세웠을 때 특정 비율 위치의 값입니다. 33.3% 분위수 아래에 전체의 약 1/3이 있습니다. |
| 중앙값(median) | 값을 크기순으로 세웠을 때 가운데 값입니다. 평균과 달리 순간적으로 튀는 값에 거의 영향을 받지 않습니다. |
| BGR / RGB | 색을 적는 순서입니다. OpenCV는 BGR(파랑·초록·빨강), matplotlib은 RGB 순서를 써서 `[..., ::-1]`로 뒤집어 표시합니다. |
| 엣지 장비 | 서버가 아닌 현장에 두고 AI 모델을 직접 돌리는 소형 컴퓨터입니다. 예: NVIDIA Jetson. |